# Electricity Demand EDA

Exploratory analysis of Germany's hourly electricity demand before model development.

**Dataset:** Open Power System Data / ENTSO-E  
**Target:** `demand_mw`

In [ ]:
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt

DATA_PATH = Path("../data/processed/germany_hourly_load.csv")

df = pd.read_csv(DATA_PATH, parse_dates=["timestamp"])
df = df.sort_values("timestamp").reset_index(drop=True)

df.head()

## Data quality

In [ ]:
print("Shape:", df.shape)
print("Start:", df["timestamp"].min())
print("End:", df["timestamp"].max())
print("Missing values:", df.isna().sum().to_dict())
print("Duplicate timestamps:", df["timestamp"].duplicated().sum())

expected = pd.date_range(
    start=df["timestamp"].min(),
    end=df["timestamp"].max(),
    freq="h",
    tz=df["timestamp"].dt.tz,
)

missing_hours = expected.difference(pd.DatetimeIndex(df["timestamp"]))
print("Missing hourly timestamps:", len(missing_hours))

df["demand_mw"].describe()

## Calendar features

In [ ]:
eda = df.copy()

eda["hour"] = eda["timestamp"].dt.hour
eda["day_name"] = eda["timestamp"].dt.day_name()
eda["month_name"] = eda["timestamp"].dt.month_name().str[:3]
eda["year"] = eda["timestamp"].dt.year

eda.head()

## Demand over time

In [ ]:
plt.figure(figsize=(14, 5))
plt.plot(eda["timestamp"], eda["demand_mw"], linewidth=0.7)
plt.title("Germany hourly electricity demand")
plt.xlabel("Time")
plt.ylabel("Demand (MW)")
plt.show()

## Average demand by hour

In [ ]:
hourly_profile = eda.groupby("hour", as_index=False)["demand_mw"].mean()

plt.figure(figsize=(10, 5))
plt.plot(hourly_profile["hour"], hourly_profile["demand_mw"], marker="o")
plt.title("Average demand by hour of day")
plt.xlabel("Hour (UTC)")
plt.ylabel("Average demand (MW)")
plt.xticks(range(0, 24, 2))
plt.show()

hourly_profile

## Average demand by weekday

In [ ]:
weekday_order = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]

weekday_profile = eda.groupby("day_name")["demand_mw"].mean().reindex(weekday_order)

plt.figure(figsize=(10, 5))
plt.bar(weekday_profile.index, weekday_profile.values)
plt.title("Average demand by day of week")
plt.xlabel("Day")
plt.ylabel("Average demand (MW)")
plt.xticks(rotation=30)
plt.show()

weekday_profile

## Hourly profile by weekday

In [ ]:
hour_weekday = eda.groupby(["day_name", "hour"])["demand_mw"].mean().reset_index()

plt.figure(figsize=(13, 6))
for day in weekday_order:
    subset = hour_weekday[hour_weekday["day_name"] == day]
    plt.plot(subset["hour"], subset["demand_mw"], label=day)

plt.title("Hourly demand profile by weekday")
plt.xlabel("Hour (UTC)")
plt.ylabel("Average demand (MW)")
plt.xticks(range(0, 24, 2))
plt.legend(ncol=2)
plt.show()

## Average demand by month

In [ ]:
month_order = ["Jan", "Feb", "Mar", "Apr", "May", "Jun", "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"]

monthly_profile = eda.groupby("month_name")["demand_mw"].mean().reindex(month_order)

plt.figure(figsize=(10, 5))
plt.bar(monthly_profile.index, monthly_profile.values)
plt.title("Average demand by month")
plt.xlabel("Month")
plt.ylabel("Average demand (MW)")
plt.show()

monthly_profile

## Demand distribution

In [ ]:
plt.figure(figsize=(10, 5))
plt.hist(eda["demand_mw"], bins=50)
plt.title("Distribution of hourly electricity demand")
plt.xlabel("Demand (MW)")
plt.ylabel("Hours")
plt.show()

## Peak and low-demand periods

In [ ]:
highest = eda.nlargest(10, "demand_mw")[["timestamp", "demand_mw"]]
lowest = eda.nsmallest(10, "demand_mw")[["timestamp", "demand_mw"]]

print("Highest-demand hours")
display(highest)

print("Lowest-demand hours")
display(lowest)

## Rolling averages

In [ ]:
eda["rolling_24h"] = eda["demand_mw"].rolling(24).mean()
eda["rolling_168h"] = eda["demand_mw"].rolling(168).mean()

sample = eda[(eda["timestamp"] >= "2019-01-01") & (eda["timestamp"] < "2019-03-01")]

plt.figure(figsize=(14, 6))
plt.plot(sample["timestamp"], sample["demand_mw"], alpha=0.4, label="Hourly demand")
plt.plot(sample["timestamp"], sample["rolling_24h"], label="24h rolling mean")
plt.plot(sample["timestamp"], sample["rolling_168h"], label="168h rolling mean")
plt.title("Demand and rolling averages")
plt.xlabel("Time")
plt.ylabel("Demand (MW)")
plt.legend()
plt.show()

## Autocorrelation at selected lags

In [ ]:
selected_lags = [1, 2, 6, 12, 24, 48, 72, 168, 336, 720]

lag_correlations = pd.DataFrame({
    "lag_hours": selected_lags,
    "autocorrelation": [eda["demand_mw"].autocorr(lag=lag) for lag in selected_lags],
})

lag_correlations

In [ ]:
plt.figure(figsize=(10, 5))
plt.bar(lag_correlations["lag_hours"].astype(str), lag_correlations["autocorrelation"])
plt.title("Demand autocorrelation at selected lags")
plt.xlabel("Lag (hours)")
plt.ylabel("Autocorrelation")
plt.ylim(-1, 1)
plt.show()

## Autocorrelation across two weeks

In [ ]:
acf_values = pd.DataFrame({
    "lag_hours": range(1, 337),
    "autocorrelation": [eda["demand_mw"].autocorr(lag=lag) for lag in range(1, 337)],
})

plt.figure(figsize=(14, 5))
plt.plot(acf_values["lag_hours"], acf_values["autocorrelation"])
plt.axvline(24, linestyle="--", alpha=0.6, label="24 hours")
plt.axvline(168, linestyle="--", alpha=0.6, label="168 hours")
plt.title("Autocorrelation of hourly electricity demand")
plt.xlabel("Lag (hours)")
plt.ylabel("Autocorrelation")
plt.legend()
plt.show()

## Key observations

- Electricity demand shows strong hourly and weekly structure.
- Weekday demand is higher than weekend demand.
- Demand varies across months.
- The series is highly autocorrelated at short lags.
- Weekly lag (`168` hours) is especially strong and supports testing weekly historical context.